# GPU Training for Parkinson's Disease Detection

This notebook downloads the dataset from the repo (or uses the already‑committed CSV), builds a tiny VGG‑style model, trains it on a GPU, and saves the trained checkpoint back to the `models/` folder of the repository.

In [13]:
# 1️⃣ Install git (already present) and clone the repo
!git clone https://github.com/Aadhithya-010/Parkinson-s-Disease-Detection.git /content/parkinsons-disease-detection
%cd /content/parkinsons-disease-detection
!git pull origin main   # or the branch you are using
# 2️⃣ Verify that the data file exists
import os, pathlib
data_path = pathlib.Path("/content/parkinsons-disease-detection/data/parkinsons_updrs.csv")
print("File exists?" , data_path.is_file())
print("Size (KB):", data_path.stat().st_size/1024 if data_path.is_file() else "N/A")

fatal: destination path '/content/parkinsons-disease-detection' already exists and is not an empty directory.
/content/parkinsons-disease-detection
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 283.73 KiB | 3.15 MiB/s, done.
From https://github.com/Aadhithya-010/Parkinson-s-Disease-Detection
 * branch            main       -> FETCH_HEAD
   67f2e6a..c020f18  main       -> origin/main
Updating 67f2e6a..c020f18
Fast-forward
 data/parkinsons_updrs.csv | 5878 ++++++++++++++++++++++++++++++++++++++++++++-
 1 file changed, 5875 insertions(+), 3 deletions(-)
File exists? True
Size (KB): 889.9033203125


In [14]:
# Install dependencies
!pip -q install tensorflow pandas scikit-learn matplotlib seaborn

In [18]:
import pandas as pd, numpy as np, os
# Path to the dataset inside the repo (mounted automatically when we clone)
data_path = '/content/parkinsons-disease-detection/data/parkinsons_updrs.csv'
df = pd.read_csv(data_path)
df.head()

,subject#,age,sex,test_time,motor_UPDRS,total_UPDRS,Jitter(%),Jitter(Abs),Jitter:RAP,Jitter:PPQ5,...,Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,Shimmer:APQ11,Shimmer:DDA,NHR,HNR,RPDE,DFA,PPE
0,1,72,0,5.6431,28.199,34.398,0.00662,0.000034,0.00401,0.00317,...,0.230,0.01438,0.01309,0.01662,0.04314,0.014290,21.640,0.41888,0.54842,0.16006
1,1,72,0,12.6660,28.447,34.894,0.00300,0.000017,0.00132,0.00150,...,0.179,0.00994,0.01072,0.01689,0.02982,0.011112,27.183,0.43493,0.56477,0.10810
2,1,72,0,19.6810,28.695,35.389,0.00481,0.000025,0.00205,0.00208,...,0.181,0.00734,0.00844,0.01458,0.02202,0.020220,23.047,0.46222,0.54405,0.21014
3,1,72,0,25.6470,28.905,35.810,0.00528,0.000027,0.00191,0.00264,...,0.327,0.01106,0.01265,0.01963,0.03317,0.027837,24.445,0.48730,0.57794,0.33277
4,1,72,0,33.6420,29.187,36.375,0.00335,0.000020,0.00093,0.00130,...,0.176,0.00679,0.00929,0.01819,0.02036,0.011625,26.126,0.47188,0.56122,0.19361


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Binary label (total_UPDRS > 15)
df['label'] = (df['total_UPDRS'] > 15).astype(int)
X = df.drop(columns=['subject#','test_time','total_UPDRS','label']).values.astype(float)
y = df['label'].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
print('Train shape', X_train.shape, 'Val shape', X_val.shape)

Train shape (4700, 19) Val shape (1175, 19)


In [20]:
import tensorflow as tf
def reshape_features(x):
    x = tf.expand_dims(x, -1)
    return tf.reshape(x, (-1, 2, 11, 1))
inputs = tf.keras.Input(shape=(X_train.shape[1],), name='features')
x = tf.keras.layers.Lambda(reshape_features)(inputs)
x = tf.keras.layers.Conv2D(32,(3,3),padding='same')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.MaxPooling2D((2,2))(x)
x = tf.keras.layers.Conv2D(64,(3,3),padding='same')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(64,activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1,activation='sigmoid')(x)
model = tf.keras.Model(inputs,outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                          loss='binary_crossentropy',
                          metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 19)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 2, 11, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 2, 11, 32)      │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2, 11, 32)      │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 2, 11, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 1, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 1, 5, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 5, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 1, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,425 (91.50 KB)

 Trainable params: 23,233 (90.75 KB)

 Non-trainable params: 192 (768.00 B)

In [23]:
# -------------------------------------------------
# 0️⃣  Determine the feature count and reshape dimensions
num_features = X_train.shape[1]           # -> 19 for the full UCI dataset
if num_features % 2 == 0:                # try to keep a height of 2 when possible
    h, w = 2, num_features // 2
else:
    h, w = 1, num_features               # fallback to a single‑row “image”

print(f"Feature shape: ({num_features},) → reshape to ({h}, {w}, 1)")

def reshape_features(x):
    # x: (batch, num_features)
    x = tf.expand_dims(x, -1)               # (batch, num_features, 1)
    x = tf.reshape(x, (-1, h, w, 1))        # (batch, h, w, 1)
    return x
# -------------------------------------------------
# 1️⃣  Build the model (use padding='same' on the pooling layer)
inputs = tf.keras.Input(shape=(num_features,), name='features')
x = tf.keras.layers.Lambda(reshape_features)(inputs)

# Tiny VGG‑like block
x = tf.keras.layers.Conv2D(32, (3, 3), padding='same')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)

# ----> 2️⃣  Pooling – choose one of the two variants below ----
# Variant A (simplest): keep pool size (2,2) but add padding='same'
x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), padding='same')(x)

# Variant B (dynamic pool size):
# if h > 1:
#     pool = tf.keras.layers.MaxPooling2D(pool_size=(2, 2), padding='valid')
# else:
#     pool = tf.keras.layers.MaxPooling2D(pool_size=(1, 2), padding='valid')
# x = pool(x)
# -------------------------------------------------

x = tf.keras.layers.Conv2D(64, (3, 3), padding='same')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()
# -------------------------------------------------

Feature shape: (19,) → reshape to (1, 19, 1)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ features (InputLayer)           │ (None, 19)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 1, 19, 1)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 1, 19, 32)      │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1, 19, 32)      │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 1, 19, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 1, 10, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 1, 10, 64)      │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 1, 10, 64)      │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 1, 10, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,425 (91.50 KB)

 Trainable params: 23,233 (90.75 KB)

 Non-trainable params: 192 (768.00 B)

In [24]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    verbose=2
)

Epoch 1/30
147/147 - 14s - 95ms/step - accuracy: 0.9245 - loss: 0.2121 - val_accuracy: 0.9140 - val_loss: 0.2932
Epoch 2/30
147/147 - 1s - 4ms/step - accuracy: 0.9449 - loss: 0.1260 - val_accuracy: 0.9140 - val_loss: 0.4305
Epoch 3/30
147/147 - 1s - 4ms/step - accuracy: 0.9581 - loss: 0.1041 - val_accuracy: 0.9166 - val_loss: 0.2029
Epoch 4/30
147/147 - 1s - 4ms/step - accuracy: 0.9630 - loss: 0.0913 - val_accuracy: 0.9566 - val_loss: 0.1058
Epoch 5/30
147/147 - 1s - 6ms/step - accuracy: 0.9649 - loss: 0.0819 - val_accuracy: 0.9515 - val_loss: 0.1086
Epoch 6/30
147/147 - 1s - 6ms/step - accuracy: 0.9717 - loss: 0.0754 - val_accuracy: 0.9813 - val_loss: 0.0629
Epoch 7/30
147/147 - 1s - 6ms/step - accuracy: 0.9683 - loss: 0.0765 - val_accuracy: 0.9779 - val_loss: 0.0728
Epoch 8/30
147/147 - 1s - 4ms/step - accuracy: 0.9723 - loss: 0.0682 - val_accuracy: 0.9804 - val_loss: 0.0741
Epoch 9/30
147/147 - 1s - 4ms/step - accuracy: 0.9730 - loss: 0.0653 - val_accuracy: 0.9821 - val_loss: 0.0789

In [25]:
# Save the trained model
model.save('models/best_trained.keras')

# Verify the file was written
!ls -lh models/

# Commit and push the notebook + model
!git add parkinsons_gpu_training.ipynb models/best_trained.keras
!git commit -m "Add trained checkpoint and updated notebook"
!git push origin main          # use your branch name if not `main`

total 344K
-rw-r--r-- 1 root root  132 Nov 27 05:31 best_cnn.keras
-rw-r--r-- 1 root root 333K Nov 27 05:54 best_trained.keras
-rw-r--r-- 1 root root  134 Nov 27 05:31 best_vgg16.keras
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@c77c23ad2757.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
# Save model into the repo's models folder
model_path = '/content/parkinsons-disease-detection/models/best_trained.keras'
model.save(model_path)
print('Model saved to', model_path)

In [ ]:
# Download the checkpoint so you can push it back to GitHub
from google.colab import files
files.download(model_path)